In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── IRLS Configuration ─────────────────────────
IRLS_C                 = 1.345
IRLS_BASE_THRESHOLD    = 3.5
IRLS_MIN_ABS_DEV       = 3.0
IRLS_MIN_POINTS        = 10
IRLS_MAX_ITER          = 50
IRLS_L2_PENALTY        = 1e-4
IV_MIN                 = 2.0
IV_MAX                 = 150.0

def enrich_data(df):
    df = df.copy()
    if 'strike' in df.columns and 'strikePrice' not in df.columns:
        df.rename(columns={'strike': 'strikePrice'}, inplace=True)
    if 'contractSymbol' in df.columns:
        extracted = df['contractSymbol'].astype(str).str.extract(r'(\d{6})([CP])')
        if not extracted.empty and extracted.shape[1] == 2:
            df['expiry_str'] = extracted[0]
            df['type_str'] = extracted[1]
            df['expiry_dt'] = pd.to_datetime(df['expiry_str'], format='%y%m%d', errors='coerce')
            df['is_put'] = (df['type_str'] == 'P').astype(int)
            if 'lastTradeDate' in df.columns:
                df['trade_date_dt'] = pd.to_datetime(df['lastTradeDate'], utc=True).dt.tz_convert(None)
                df['daysToExpiration'] = (df['expiry_dt'] - df['trade_date_dt']).dt.days
    return df


Please upload `aapl_20260320_calls_264_58.csv` (or edit the path below to point to your CSV file).

In [ ]:
# Load options data sample 
csv_file = "options_data/2026-02-20/aapl_20260320_calls_264_58.csv"  # Modify path if needed in Colab
try:
    df = pd.read_csv(csv_file)
except FileNotFoundError:
    print(f"File '{csv_file}' not found. Please upload it directly to Colab or adjust the path.")
    df = pd.DataFrame()  # Prevent further errors during cell execution

if not df.empty:
    df = enrich_data(df)

    # Filter criteria matching process_file()
    if 'lastPrice' in df.columns:
        df = df[df['lastPrice'] > 0.01]
    if 'volume' in df.columns:
        df = df[df['volume'] >= 10]
    if 'impliedVolatility' in df.columns:
        df = df[(df['impliedVolatility'] > IV_MIN) & (df['impliedVolatility'] < IV_MAX)]
    if 'daysToExpiration' in df.columns:
        df = df[df['daysToExpiration'] >= 0.9]

    needed = {'strikePrice', 'underlyingPriceAtTrade', 'daysToExpiration', 'impliedVolatility'}
    work = df.copy()
    for c in needed:
        work[c] = pd.to_numeric(work[c], errors='coerce')
    work = work.dropna(subset=list(needed))
    work = work[work['daysToExpiration'] > 0]
    work = work[work['underlyingPriceAtTrade'] > 0]


In [ ]:
if 'work' in locals() and not work.empty and len(work) >= IRLS_MIN_POINTS:
    # Fit the theoretical IRLS surface for the option curve
    m = np.log(work['strikePrice'] / work['underlyingPriceAtTrade']).values
    t = np.sqrt(work['daysToExpiration']).values
    y = work['impliedVolatility'].values

    X = np.column_stack([
        np.ones(len(m)),   
        m,                  
        m ** 2,             
        t,                  
        t ** 2,             
        m * t,              
        m ** 2 * t,         
    ])

    w = np.exp(-2.0 * np.abs(m))
    beta = np.zeros(X.shape[1])

    I_reg = np.eye(X.shape[1])
    I_reg[0, 0] = 0

    for _ in range(IRLS_MAX_ITER):
        W = np.diag(w)
        XtW = X.T @ W
        try:
            beta_new = np.linalg.solve(XtW @ X + IRLS_L2_PENALTY * I_reg, XtW @ y)
        except np.linalg.LinAlgError:
            print("Singular matrix! Couldn't fit IRLS.")
            break

        if np.max(np.abs(beta_new - beta)) < 1e-6:
            beta = beta_new
            break
        beta = beta_new

        residuals = y - X @ beta
        mad = np.median(np.abs(residuals))
        mad_scale = 1.4826 * mad if mad > 1e-10 else 1.0  
        r = residuals / mad_scale

        abs_r = np.abs(r)
        w = np.where(abs_r < IRLS_C, 1.0, IRLS_C / np.maximum(abs_r, 1e-10))

    theoretical_iv = X @ beta
    work['Theoretical_IV'] = theoretical_iv
else:
    print("Not enough points to fit the curve or data wasn't loaded.")


In [ ]:
if 'work' in locals() and 'Theoretical_IV' in work.columns:
    # Plotting Real IV vs Theoretical IV Curve
    work = work.sort_values(by='strikePrice')

    plt.figure(figsize=(10, 6))
    plt.scatter(work['strikePrice'], work['impliedVolatility'], label='Real IV (Data)', color='blue', alpha=0.6)
    plt.plot(work['strikePrice'], work['Theoretical_IV'], label='Theoretical Option Curve (IRLS)', color='red', linewidth=2)

    plt.title('Implied Volatility: Real vs Theoretical (IRLS surface)')
    plt.xlabel('Strike Price')
    plt.ylabel('Implied Volatility')
    plt.legend()
    plt.grid(True)
    plt.show()
